# Cross-Domain Plant Disease Detection
## 7th Semester Deep Learning Project

Welcome to the interactive walkthrough of our model training pipeline. 
In this notebook, we will:
1. Explore the **Domain Gap** between Lab (PlantVillage) and Field (PlantDoc) data.
2. Train a **Baseline ResNet18** model.
3. Apply **Domain Adaptation (Fine-Tuning)** to improve real-world performance.

### Step 1: Imports and Device Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../model'))
import torch
import torch.nn as nn
import torch.optim as optim
from dataset import get_dataloaders
from model import get_model, freeze_backbone, unfreeze_all
from train_baseline import train_model

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### Step 2: Data Loading & The Domain Gap
We load data from two different domains. Notice how we use heavy augmentations on the training data in `dataset.py` to artificially introduce real-world noise (blur, color jitter).

In [ ]:
# Load Source Domain (Lab Data)
source_dir = '../data/PlantVillage'
source_train, source_val, _, classes = get_dataloaders(source_dir, batch_size=16)

# Load Target Domain (Field Data)
target_dir = '../data/PlantDoc'
target_train, target_val, _, _ = get_dataloaders(target_dir, batch_size=16)

print(f"Classes: {classes}")
print(f"Source training batches: {len(source_train)}")
print(f"Target training batches: {len(target_train)}")

### Step 3: Baseline Training (Source Domain Only)
We train a ResNet18 model *only* on the clean, lab-captured PlantVillage dataset. We expect this to perform poorly on the target domain.

In [ ]:
baseline_model = get_model(num_classes=len(classes), pretrained=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(baseline_model.parameters(), lr=0.001)

print("Training Baseline Model...")
# We use 2 epochs just for quick testing in this notebook
baseline_model = train_model(baseline_model, {'train': source_train, 'val': source_val}, criterion, optimizer, num_epochs=2, device=device)

torch.save(baseline_model.state_dict(), '../baseline_model.pth')

### Step 4: Domain Adaptation (Fine-Tuning)
To fix the domain gap, we take our baseline model and fine-tune it on the noisy PlantDoc dataset.
We do this in two stages:
1. **Stage 1**: Freeze the backbone feature extractor and train *only* the final classification layer. This prevents us from immediately destroying the useful features learned from the large source dataset.
2. **Stage 2**: Unfreeze everything and train with a very small learning rate to gently adapt the entire network to the target domain.

In [ ]:
# Load baseline
adapted_model = get_model(num_classes=len(classes), pretrained=False).to(device)
adapted_model.load_state_dict(torch.load('../baseline_model.pth', weights_only=True))

# STAGE 1: Freeze Backbone
adapted_model = freeze_backbone(adapted_model)
optimizer_stage1 = optim.Adam(adapted_model.fc.parameters(), lr=0.001)
print("Stage 1: Fine-tuning final layer...")
adapted_model = train_model(adapted_model, {'train': target_train, 'val': target_val}, criterion, optimizer_stage1, num_epochs=1, device=device)

# STAGE 2: Unfreeze All
adapted_model = unfreeze_all(adapted_model)
optimizer_stage2 = optim.Adam(adapted_model.parameters(), lr=1e-5) # Tiny learning rate!
print("Stage 2: Full network adaptation...")
adapted_model = train_model(adapted_model, {'train': target_train, 'val': target_val}, criterion, optimizer_stage2, num_epochs=1, device=device)

torch.save(adapted_model.state_dict(), '../domain_adapted_model.pth')

### Step 5: Evaluation
You can now run `evaluate.py` or use the Streamlit app to visually compare how much better the `adapted_model` performs on field data compared to the `baseline_model`!